In [24]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers


import os
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime, date, timedelta


# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.ingester.IngesterClass import Ingester
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [26]:
class EmissionDistributionIngester(Ingester):
    def __init__(self, arguments):
        super().__init__(arguments)
        self.table = KPI_EmissionDistribution

    def update_check(self):
        customer_name = self.customer_info['Name']
        customer_id = self.customer_info['CustomerId']
        customer_db = self.customer_info['DBLocation']

        self.Logger.info(f"Processing customer: {customer_name}")
        
        #Query the reports from KPI_EmissionSources
        query_kpi_emission_sources = f"""SELECT DISTINCT ReportId FROM KPI_EmissionDistribution WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}')"""
        reports_kpi_emission_sources = Query(query = query_kpi_emission_sources).execute(KPIHub_Conn)
        num_reports_kpi_emission_sources = len(reports_kpi_emission_sources)

        #Query the reports from KPI_ReportSummary
        query_kpi_report = f"""SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}'"""
        reports_kpi_hub = Query(query = query_kpi_report).execute(KPIHub_Conn)
 
        num_reports_kpi_hub = len(reports_kpi_hub)

        #Check if there are new reports
        reports_into = reports_kpi_hub[~reports_kpi_hub['ReportId'].isin(reports_kpi_emission_sources['ReportId'])]
        reports_deleted = reports_kpi_emission_sources[~reports_kpi_emission_sources['ReportId'].isin(reports_kpi_hub['ReportId'])]

        self.data['reports_into'] = reports_into.copy()
        self.data['reports_deleted'] = reports_deleted.copy()
        self.data['num_reports_kpi_emission_sources'] = num_reports_kpi_emission_sources
        self.data['num_reports_kpi_hub'] = num_reports_kpi_hub

        if (num_reports_kpi_hub > 0):
            self.Logger.info(f"Number of reports in KPI_ReportSummary: {num_reports_kpi_hub}")
            if(num_reports_kpi_emission_sources == 0):
                #No reports in the KPI_EmissionSources
                self.Logger.info("No reports in KPIHub, starting from the beginning")
                self.check_flag = True
            else:
                #Reports in the KPIHub
                self.Logger.info(f"Number of reports in KPI_EmissionDistribution: {num_reports_kpi_emission_sources}")
                if len(reports_into) > 0 or len(reports_deleted) > 0:
                    self.Logger.info(f"Number of new reports into the KPI_EmissionDistribution: {len(reports_into)}")
                    self.Logger.info(f"Number of deleted reports in the KPI_ReportSummary: {len(reports_deleted)}")
                    self.check_flag = True
                else:
                    self.Logger.info("No new reports into the KPI_EmissionDistribution or deleted reports in the KPI_EmissionDistribution")
                    self.check_flag = False
        else:
            self.Logger.info("No reports in KPI_ReportSummary")
            self.check_flag = False

    def query_data(self):
        if self.check_flag and len(self.data['reports_into']) > 0:
            reports_to_query = self.data['reports_into'].copy()
            reports_to_query.db.set_query(get_emission_soruces_for_RER(report_table = '#TempReports'))
            emission_sources = reports_to_query.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'ReportId', temp_table_name = '#TempReports')
            emission_sources['LastUpdated'] = datetime.now()
            self.data['output'] = emission_sources
        else:
            self.Logger.info(f"No reports found, skipping")

    def push_data(self):
        super().push_data(primary_key = 'EmissionSourceId')

 
    def sanity_check(self):
        super().sanity_check()
        #Get all the reports from the KPI_SurveySummary table
        df_surveys = Query(query = f"SELECT DISTINCT ReportId FROM KPI_EmissionDistribution WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}')").execute(KPIHub_Conn)
        self.Logger.info(f"Total number of unique reports from KPI_EmissionDistribution: {len(df_surveys)}")



In [27]:
#Testing the class
customer_list = get_customer_list(KPIHub_Conn)
for idx,customer_info in customer_list.iterrows():
    print(customer_info['Name'])
    ingester = EmissionDistributionIngester(arguments = {'conn': KPIHub_Conn})
    ingester.set_customer_info(customer_info)
    ingester.update_check()
    ingester.query_data()
    ingester.push_data()
    ingester.delete_data()
    ingester.sanity_check()


Wales and West Utilities


APRETIGAS
ADRIGAS
EDMA
M Reti
Sei Mantova
CPL CONCORDIA
SIG


--- Logging error ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.9/logging/__init__.py", line 1086, in emit
    stream.write(msg + self.terminator)
  File "/home/sandbox/personal-repos/packages/locallib/slack/IOSlack.py", line 29, in write
    self.app.client.chat_postMessage(channel=self.channel, text=s)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/client.py", line 2795, in chat_postMessage
    return self.api_call("chat.postMessage", json=kwargs)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 169, in api_call
    return self._sync_send(api_url=api_url, req_args=req_args)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 200, in _sync_send
    return self._urllib_api_call(
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 323, in _urllib_api_call
    r

RETEGAS BARI spa


--- Logging error ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.9/logging/__init__.py", line 1086, in emit
    stream.write(msg + self.terminator)
  File "/home/sandbox/personal-repos/packages/locallib/slack/IOSlack.py", line 29, in write
    self.app.client.chat_postMessage(channel=self.channel, text=s)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/client.py", line 2795, in chat_postMessage
    return self.api_call("chat.postMessage", json=kwargs)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 169, in api_call
    return self._sync_send(api_url=api_url, req_args=req_args)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 200, in _sync_send
    return self._urllib_api_call(
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 323, in _urllib_api_call
    r

...

--- Logging error ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.9/logging/__init__.py", line 1086, in emit
    stream.write(msg + self.terminator)
  File "/home/sandbox/personal-repos/packages/locallib/slack/IOSlack.py", line 29, in write
    self.app.client.chat_postMessage(channel=self.channel, text=s)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/client.py", line 2795, in chat_postMessage
    return self.api_call("chat.postMessage", json=kwargs)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 169, in api_call
    return self._sync_send(api_url=api_url, req_args=req_args)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 200, in _sync_send
    return self._urllib_api_call(
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 323, in _urllib_api_call
    r

RETI Distribuzione


--- Logging error ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.9/logging/__init__.py", line 1086, in emit
    stream.write(msg + self.terminator)
  File "/home/sandbox/personal-repos/packages/locallib/slack/IOSlack.py", line 29, in write
    self.app.client.chat_postMessage(channel=self.channel, text=s)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/client.py", line 2795, in chat_postMessage
    return self.api_call("chat.postMessage", json=kwargs)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 169, in api_call
    return self._sync_send(api_url=api_url, req_args=req_args)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 200, in _sync_send
    return self._urllib_api_call(
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 323, in _urllib_api_call
    r

Stedin


--- Logging error ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.9/logging/__init__.py", line 1086, in emit
    stream.write(msg + self.terminator)
  File "/home/sandbox/personal-repos/packages/locallib/slack/IOSlack.py", line 29, in write
    self.app.client.chat_postMessage(channel=self.channel, text=s)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/client.py", line 2795, in chat_postMessage
    return self.api_call("chat.postMessage", json=kwargs)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 169, in api_call
    return self._sync_send(api_url=api_url, req_args=req_args)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 200, in _sync_send
    return self._urllib_api_call(
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/slack_sdk/web/base_client.py", line 323, in _urllib_api_call
    r